# S2G 01 - AndinaLog IoT Gold

Construye la tabla Gold predictiva de desviacion termica a partir de IoT Silver.
Una fila Gold por lectura Silver. Los predictores usan solo informacion disponible
en el instante de prediccion; el futuro se utiliza unicamente para construir la
etiqueta `desviacion_proximos_60min_flag`.

In [1]:
import json
import os
import platform
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)


def detectar_raiz():
    relativa = Path('datos/silver/andinalog_iot_telemetry_silver.csv')
    candidatas = []
    if os.getenv('ANDINALOG_ROOT'):
        candidatas.append(Path(os.environ['ANDINALOG_ROOT']))
    cwd = Path.cwd().resolve()
    candidatas.extend([cwd, *cwd.parents])
    contenido = Path('/content').resolve()
    candidatas.extend([contenido, *contenido.parents])
    visitadas = set()
    for candidata in candidatas:
        normalizada = candidata.resolve()
        if normalizada in visitadas:
            continue
        visitadas.add(normalizada)
        if (normalizada / relativa).exists():
            return normalizada
    raise FileNotFoundError('No se encontro datos/silver/andinalog_iot_telemetry_silver.csv')


CONFIG = {
    'rutas': {
        'silver': 'datos/silver/andinalog_iot_telemetry_silver.csv',
        'gold': 'datos/gold/andinalog_iot_modelado_gold.csv',
        'informe': 'informes/silver_gold/Informe_S2G_01_IoT_Gold.md',
        'notebook': 'notebooks/silver_gold/01_iot_gold/S2G_01_AndinaLog_IoT_Gold.ipynb',
    },
    'columnas': {
        'timestamp': 'timestamp',
        'entidad': 'viaje_id',
        'orden': 'order_id',
        'camion': 'camion_id',
        'producto': 'producto_id',
        'id_lectura': '_fila_bronze',
        'temperatura': 'temperatura_cabina_c',
        'humedad': 'humedad_cabina_pct',
        'flag_actual': 'desviacion_termica_flag',
        'flag_silver': 'desviacion_proximos_60min_flag',
    },
    'patron_entidad': r'^VIA-\d{5}$',
    'ventana_historica_min': 30,
    'horizonte_objetivo_min': 60,
    'frecuencia_esperada_min': 30,
    'tolerancia_busqueda_min': 15,
    'tolerancia_cobertura_min': 15,
    'metodo_temperatura_anterior': (
        'Ultima lectura con timestamp estrictamente anterior a t dentro del mismo viaje, '
        'sin tolerancia de distancia; la primera lectura de cada viaje queda nula.'
    ),
    'metodo_variacion': (
        'Busqueda temporal dentro del mismo viaje de la lectura mas reciente con timestamp en '
        '[t-45, t-30] minutos; variacion = temperatura actual menos temperatura de esa lectura. '
        'Sin lectura en ese rango la variacion queda nula.'
    ),
    'criterio_cobertura': (
        'La ventana objetivo es evaluable si la ultima lectura futura dentro de (t, t+60] alcanza '
        'al menos t+45 minutos. Sin esa cobertura el objetivo queda nulo y la bandera en False.'
    ),
    'definicion_objetivo': (
        '1 si existe al menos una desviacion_termica_flag igual a 1 en (t, t+60] dentro del mismo viaje; '
        '0 solo con cobertura suficiente y sin desviacion; nulo sin cobertura suficiente.'
    ),
    'semilla': None,
    'nota_semilla': 'El pipeline es determinista y no usa aleatoriedad; la semilla no aplica.',
    'evidencia_entrada': {
        'bronze': 'datos/bronze/andinalog_iot_telemetry.csv',
        'cuarentena': 'datos/quarantine/andinalog_iot_telemetry_quarantine.csv',
        'informe_silver': 'informes/bronze_silver/Informe_B2S_04_IoT_Telemetry.md',
        'notebook_silver': 'notebooks/bronze_silver/04_iot_telemetry/B2S_04_AndinaLog_IoT_Telemetry.ipynb',
        'patron_conciliacion': r'Conciliacion: Bronze (\d+) = Silver (\d+) \+ cuarentena (\d+)',
    },
    'motivos': {
        'con_antecedente': '',
        'primera_lectura': 'primera_lectura_del_viaje',
        'sin_lectura_30min': 'sin_lectura_en_[t-45,t-30]_minutos',
    },
}
CONFIG['clave_lectura'] = [CONFIG['columnas']['entidad'], CONFIG['columnas']['timestamp']]
CONFIG['columnas_derivadas'] = [
    'temperatura_anterior',
    'ts_temperatura_anterior',
    'variacion_temperatura_30min',
    'ts_base_variacion',
]
CONFIG['columnas_salida'] = [
    CONFIG['columnas']['id_lectura'],
    CONFIG['columnas']['timestamp'],
    CONFIG['columnas']['entidad'],
    CONFIG['columnas']['orden'],
    CONFIG['columnas']['camion'],
    CONFIG['columnas']['producto'],
    CONFIG['columnas']['temperatura'],
    CONFIG['columnas']['humedad'],
    CONFIG['columnas']['flag_actual'],
    'temperatura_anterior',
    'ts_temperatura_anterior',
    'variacion_temperatura_30min',
    'ts_base_variacion',
    'motivo_temperatura_anterior',
    'motivo_variacion_30min',
    CONFIG['columnas']['flag_silver'],
    'ventana_objetivo_evaluable',
    'objetivo_coincide_con_silver',
    'calidad_estado',
    'calidad_motivo',
]

RAIZ = detectar_raiz()
EJECUTADO_UTC = datetime.now(timezone.utc).isoformat()
PATHS = {clave: RAIZ / valor for clave, valor in CONFIG['rutas'].items()}
PATHS['gold'].parent.mkdir(parents=True, exist_ok=True)
PATHS['informe'].parent.mkdir(parents=True, exist_ok=True)

print('Raiz detectada:', RAIZ)
print('Ejecucion UTC:', EJECUTADO_UTC)
print('Entidad:', CONFIG['columnas']['entidad'], '| Horizonte:', CONFIG['horizonte_objetivo_min'], 'min')
print('Semilla:', CONFIG['semilla'], '|', CONFIG['nota_semilla'])

Raiz detectada: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2
Ejecucion UTC: 2026-09-25T17:11:58.718685+00:00
Entidad: viaje_id | Horizonte: 60 min
Semilla: None | El pipeline es determinista y no usa aleatoriedad; la semilla no aplica.


In [2]:
silver = pd.read_csv(PATHS['silver'], dtype=str, keep_default_na=False)

columnas = list(silver.columns)
faltantes = [
    nombre for nombre in CONFIG['columnas'].values()
    if nombre not in columnas and nombre != CONFIG['columnas']['flag_silver']
]
if faltantes:
    raise ValueError(f'Columnas requeridas ausentes en IoT Silver: {faltantes}')

print('Forma Silver:', silver.shape)
print()
print('Columnas de Silver:', len(columnas))
for indice, nombre in enumerate(columnas):
    print(f'  {indice:>2}: {nombre}')
print()
print('Tipos de dato crudos (Silver se lee como texto para no perder vacios):',
      sorted({str(tipo) for tipo in silver.dtypes}))
print()

nulos = pd.DataFrame({
    'vacias': (silver == '').sum(),
})
nulos = nulos[nulos['vacias'] > 0]
print('Columnas con valores vacios en Silver:')
print(nulos.to_string() if len(nulos) else '  ninguna')
print()

clave = CONFIG['clave_lectura']
print('Clave de lectura:', clave)
print('Filas duplicadas en la clave de lectura:', int(silver.duplicated(clave).sum()))
print('Identificador de lectura unico:', bool(silver[CONFIG['columnas']['id_lectura']].is_unique))
print()

ts_texto = silver[CONFIG['columnas']['timestamp']]
print('Sufijos de zona horaria presentes:', sorted({texto[-6:] for texto in ts_texto}))
ts_utc = pd.to_datetime(ts_texto, format='ISO8601', utc=True)
print('Timestamp minimo (UTC):', ts_utc.min())
print('Timestamp maximo (UTC):', ts_utc.max())
print('Valores de timestamp no parseables:', int(ts_utc.isna().sum()))
print()

entidad = CONFIG['columnas']['entidad']
print('Entidad:', entidad)
print('Entidades distintas:', int(silver[entidad].nunique()))
print('Entidades fuera del patron', CONFIG['patron_entidad'] + ':',
      int((~silver[entidad].str.fullmatch(CONFIG['patron_entidad'])).sum()))
print('Ordenes distintos:', int(silver[CONFIG['columnas']['orden']].nunique()))
print('Camiones distintos:', int(silver[CONFIG['columnas']['camion']].nunique()))
print('Productos distintos:', int(silver[CONFIG['columnas']['producto']].nunique()))
print('Ordenes por viaje (debe ser 1):',
      int(silver.groupby(entidad)[CONFIG['columnas']['orden']].nunique().max()))
print()

marco = pd.DataFrame({'entidad': silver[entidad].to_numpy(), 'ts': ts_utc.to_numpy()})
ordenado = marco.sort_values(['entidad', 'ts'], kind='stable')
gaps = ordenado.groupby('entidad')['ts'].diff().dt.total_seconds().div(60).dropna()
print('Frecuencia observada: intervalos internos por entidad, en minutos')
print(gaps.value_counts().sort_index().to_string())
print('Intervalo minimo:', float(gaps.min()), '| intervalo maximo:', float(gaps.max()))
print('Intervalo mediano:', float(gaps.median()))
print('Lecturas con intervalo mayor a la cadencia esperada:',
      int((gaps > CONFIG['frecuencia_esperada_min']).sum()))
print('Mediana observada coincide con la cadencia de config:',
      float(gaps.median()) == float(CONFIG['frecuencia_esperada_min']))
print('Intervalos no multiplos de la cadencia esperada:',
      int((gaps % CONFIG['frecuencia_esperada_min'] != 0).sum()))
print()

print('Distribucion de la desviacion actual:',
      silver[CONFIG['columnas']['flag_actual']].value_counts().to_dict())
print('Distribucion de la etiqueta que ya trae Silver:',
      silver[CONFIG['columnas']['flag_silver']].value_counts().to_dict())
errores_bloqueantes = int(silver['errores_bloqueantes'].ne('').sum()) if 'errores_bloqueantes' in silver.columns else 0
print('Filas Silver con errores bloqueantes:', errores_bloqueantes)
assert errores_bloqueantes == 0, f'Compuerta previa fallida: Silver trae {errores_bloqueantes} errores bloqueantes'

Forma Silver: (28448, 78)

Columnas de Silver: 78
   0: timestamp
   1: viaje_id
   2: order_id
   3: camion_id
   4: producto_id
   5: temperatura_cabina_c
   6: temp_unit
   7: humedad_cabina_pct
   8: desviacion_termica_flag
   9: desviacion_proximos_60min_flag
  10: _fila_bronze
  11: errores_bloqueantes
  12: banderas_informativas
  13: motivos_transformacion
  14: motivos_imputacion
  15: timestamp_original
  16: viaje_id_original
  17: order_id_original
  18: camion_id_original
  19: producto_id_original
  20: temperatura_cabina_c_original
  21: temp_unit_original
  22: humedad_cabina_pct_original
  23: desviacion_termica_flag_original
  24: desviacion_proximos_60min_flag_original
  25: viaje_id_tratado
  26: viaje_id_transformado
  27: order_id_tratado
  28: order_id_transformado
  29: camion_id_tratado
  30: camion_id_transformado
  31: producto_id_tratado
  32: producto_id_transformado
  33: temp_unit_tratado
  34: temp_unit_transformado
  35: temperatura_cabina_c_conversion_

Columnas con valores vacios en Silver:
                              vacias
humedad_cabina_pct                98
errores_bloqueantes            28448
banderas_informativas          23505
motivos_imputacion             28448
humedad_cabina_pct_original       98
humedad_cabina_pct_tratado        98
intervalo_fuente_min            1194
desviacion_termica_calculada    3109
imputacion_metodo              28448
imputacion_motivo              28448

Clave de lectura: ['viaje_id', 'timestamp']
Filas duplicadas en la clave de lectura: 0
Identificador de lectura unico: True

Sufijos de zona horaria presentes: ['+00:00']
Timestamp minimo (UTC): 2026-08-01 04:10:00+00:00
Timestamp maximo (UTC): 2026-08-31 15:11:00+00:00
Valores de timestamp no parseables: 0

Entidad: viaje_id
Entidades distintas: 1200
Entidades fuera del patron ^VIA-\d{5}$: 0
Ordenes distintos: 1200
Camiones distintos: 30
Productos distintos: 60
Ordenes por viaje (debe ser 1): 1

Frecuencia observada: intervalos internos por entid

## Compuerta previa a Gold

La skill de Gold exige como entrada minima IoT Silver con evidencia de conciliacion y de
auditoria. Esta celda verifica, con los archivos actuales, que la evidencia existe y que la
conciliacion Bronze = Silver + cuarentena se cumple; no reproduce cifras del informe anterior.

In [3]:
evidencia = {clave: RAIZ / valor for clave, valor in CONFIG['evidencia_entrada'].items()
             if clave != 'patron_conciliacion'}
print('Evidencia de la etapa Bronze-Silver utilizada:')
for clave, ruta in evidencia.items():
    print(f'  {clave}: {ruta.relative_to(RAIZ).as_posix()} | existe: {ruta.exists()}'
          + (f' | bytes: {ruta.stat().st_size}' if ruta.exists() else ''))
faltantes = [ruta.relative_to(RAIZ).as_posix() for ruta in evidencia.values() if not ruta.exists()]
assert not faltantes, f'Compuerta previa fallida: falta evidencia {faltantes}'

bronze = pd.read_csv(evidencia['bronze'], usecols=[0], dtype=str, keep_default_na=False)
cuarentena = pd.read_csv(evidencia['cuarentena'], dtype=str, keep_default_na=False)
texto_informe = evidencia['informe_silver'].read_text(encoding='utf-8')
conciliacion = re.search(CONFIG['evidencia_entrada']['patron_conciliacion'], texto_informe)
print()
print('Conciliacion declarada en el informe Silver:', conciliacion.group(0) if conciliacion else 'NO ENCONTRADA')
bronze_declarado, silver_declarado, cuarentena_declarado = (
    (int(conciliacion.group(1)), int(conciliacion.group(2)), int(conciliacion.group(3)))
    if conciliacion else (None, None, None))
print('Filas Bronze leidas ahora:', len(bronze))
print('Filas Silver leidas ahora:', len(silver))
print('Filas de cuarentena leidas ahora:', len(cuarentena))
conciliacion_actual = (len(bronze), len(silver), len(cuarentena))
print('Bronze = Silver + cuarentena verificado ahora:',
      len(bronze) == len(silver) + len(cuarentena))
assert len(bronze) == len(silver) + len(cuarentena), 'Compuerta previa fallida: la conciliacion Bronze = Silver + cuarentena no se cumple'
if conciliacion is not None:
    assert conciliacion_actual == (bronze_declarado, silver_declarado, cuarentena_declarado), (
        'Compuerta previa fallida: los conteos actuales difieren de los declarados en el informe Silver')
    print('Conteos actuales identicos a los declarados en el informe Silver: True')
print('Silver con clave de lectura unica:', not silver.duplicated(CONFIG['clave_lectura']).any())
print('Silver con errores bloqueantes:', errores_bloqueantes)
print('Silver con imputaciones registradas:', int(silver['fue_imputada'].eq('True').sum())
      if 'fue_imputada' in silver.columns else 0)
print('COMPUERTA_PREVIA_OK')

Evidencia de la etapa Bronze-Silver utilizada:
  bronze: datos/bronze/andinalog_iot_telemetry.csv | existe: True | bytes: 2248555
  cuarentena: datos/quarantine/andinalog_iot_telemetry_quarantine.csv | existe: True | bytes: 276892
  informe_silver: informes/bronze_silver/Informe_B2S_04_IoT_Telemetry.md | existe: True | bytes: 4932
  notebook_silver: notebooks/bronze_silver/04_iot_telemetry/B2S_04_AndinaLog_IoT_Telemetry.ipynb | existe: True | bytes: 37814



Conciliacion declarada en el informe Silver: Conciliacion: Bronze 28920 = Silver 28448 + cuarentena 472
Filas Bronze leidas ahora: 28920
Filas Silver leidas ahora: 28448
Filas de cuarentena leidas ahora: 472
Bronze = Silver + cuarentena verificado ahora: True
Conteos actuales identicos a los declarados en el informe Silver: True
Silver con clave de lectura unica: True
Silver con errores bloqueantes: 0
Silver con imputaciones registradas: 0
COMPUERTA_PREVIA_OK


In [4]:
def preparar(df):
    out = df.copy()
    ts = pd.to_datetime(out[CONFIG['columnas']['timestamp']], format='ISO8601', utc=True)
    out['ts_utc'] = ts
    out['temperatura_c'] = pd.to_numeric(
        out[CONFIG['columnas']['temperatura']].replace('', np.nan), errors='coerce')
    out['humedad_pct'] = pd.to_numeric(
        out[CONFIG['columnas']['humedad']].replace('', np.nan), errors='coerce')
    out['desviacion_actual'] = pd.to_numeric(
        out[CONFIG['columnas']['flag_actual']].replace('', np.nan), errors='coerce')
    out['flag_silver'] = pd.to_numeric(
        out[CONFIG['columnas']['flag_silver']].replace('', np.nan), errors='coerce')
    return out.sort_values(['ts_utc', CONFIG['columnas']['entidad']], kind='stable').reset_index(drop=True)


work = silver.pipe(preparar)
work = work.sort_values([CONFIG['columnas']['entidad'], 'ts_utc'], kind='stable').reset_index(drop=True)

print('Tipos despues de la preparacion temporal:')
print(work[['ts_utc', 'temperatura_c', 'humedad_pct', 'desviacion_actual', 'flag_silver']].dtypes.to_string())
print()
print('FilasSilver:', len(silver), '| Filas en trabajo:', len(work))
print('Diferencia de filas en la preparacion:', len(work) - len(silver))
print('Lecturas con temperatura no numerica:', int(work['temperatura_c'].isna().sum()))
print('Lecturas con desviacion actual no numerica:', int(work['desviacion_actual'].isna().sum()))
print('Duplicados en la clave de lectura:', int(work.duplicated(CONFIG['clave_lectura']).sum()))

monotonicas = work.groupby(CONFIG['columnas']['entidad'])['ts_utc'].apply(
    lambda serie: bool(serie.is_monotonic_increasing))
print('Grupos con orden temporal correcto:', int(monotonicas.sum()), 'de', len(monotonicas))
print('Grupos con orden temporal incorrecto:', int((~monotonicas).sum()))

Tipos despues de la preparacion temporal:
ts_utc               datetime64[us, UTC]
temperatura_c                    float64
humedad_pct                      float64
desviacion_actual                  int64
flag_silver                        int64

FilasSilver: 28448 | Filas en trabajo: 28448
Diferencia de filas en la preparacion: 0
Lecturas con temperatura no numerica: 0
Lecturas con desviacion actual no numerica: 0
Duplicados en la clave de lectura: 0


Grupos con orden temporal correcto: 1200 de 1200
Grupos con orden temporal incorrecto: 0


In [5]:
def calcular_temperatura_anterior(df):
    out = df.copy()
    entidad = CONFIG['columnas']['entidad']
    grupo = out.groupby(entidad, sort=False)
    out['temperatura_anterior'] = grupo['temperatura_c'].shift(1)
    out['ts_temperatura_anterior'] = grupo['ts_utc'].shift(1)
    return out


work = work.pipe(calcular_temperatura_anterior)

print('Regla:', CONFIG['metodo_temperatura_anterior'])
print('Lecturas con temperatura anterior:', int(work['temperatura_anterior'].notna().sum()))
print('Lecturas sin temperatura anterior:', int(work['temperatura_anterior'].isna().sum()))
print('Lecturas sin temperatura anterior por viaje (esperado 1 por viaje):',
      int(work.groupby(CONFIG['columnas']['entidad'])['temperatura_anterior'].apply(
          lambda serie: int(serie.isna().sum())).sum()))
print('Alguna temperatura anterior posterior a t:',
      int((work['ts_temperatura_anterior'].notna() &
           (work['ts_temperatura_anterior'] >= work['ts_utc'])).sum()))

Regla:

 Ultima lectura con timestamp estrictamente anterior a t dentro del mismo viaje, sin tolerancia de distancia; la primera lectura de cada viaje queda nula.
Lecturas con temperatura anterior: 27248
Lecturas sin temperatura anterior: 1200
Lecturas sin temperatura anterior por viaje (esperado 1 por viaje): 1200
Alguna temperatura anterior posterior a t: 0


In [6]:
def buscar_en_entidad(ts_entidad, desplazamiento_min, tolerancia_min, lado):
    objetivo = ts_entidad + np.timedelta64(int(desplazamiento_min), 'm')
    limite_inferior = objetivo - np.timedelta64(int(tolerancia_min), 'm')
    limite_superior = objetivo + np.timedelta64(int(tolerancia_min), 'm')
    if lado == 'pasado':
        indice = np.searchsorted(ts_entidad, limite_superior, side='right') - 1
        valido = (indice >= 0) & (ts_entidad[np.clip(indice, 0, len(ts_entidad) - 1)] >= limite_inferior)
    else:
        indice = np.searchsorted(ts_entidad, limite_inferior, side='left')
        recortado = np.clip(indice, 0, len(ts_entidad) - 1)
        valido = (indice < len(ts_entidad)) & (ts_entidad[recortado] <= limite_superior)
    return np.where(valido, indice, -1)


def calcular_variacion_30min(df):
    out = df.copy()
    entidad = CONFIG['columnas']['entidad']
    ventana = CONFIG['ventana_historica_min']
    tolerancia = CONFIG['tolerancia_busqueda_min']
    sin_base = np.full(len(out), np.nan)
    marca = np.full(len(out), np.datetime64('NaT', 'ns'))
    ts_plano = out['ts_utc'].dt.tz_convert('UTC').dt.tz_localize(None).to_numpy('datetime64[ns]')
    for _, indices in out.groupby(entidad, sort=False).groups.items():
        indices = np.asarray(indices)
        ts_entidad = ts_plano[indices]
        orden = np.argsort(ts_entidad, kind='stable')
        indices = indices[orden]
        ts_entidad = ts_entidad[orden]
        encontrados = buscar_en_entidad(
            ts_entidad, -ventana, tolerancia, lado='pasado')
        disponibles = encontrados >= 0
        if not disponibles.any():
            continue
        sin_base[indices[disponibles]] = out['temperatura_c'].to_numpy()[indices[encontrados[disponibles]]]
        marca[indices[disponibles]] = ts_entidad[encontrados[disponibles]]
    out['temperatura_base_30min'] = sin_base
    out['ts_base_variacion'] = pd.Series(marca, index=out.index).dt.tz_localize('UTC')
    out['variacion_temperatura_30min'] = out['temperatura_c'] - out['temperatura_base_30min']
    return out


work = work.pipe(calcular_variacion_30min)

print('Regla:', CONFIG['metodo_variacion'])
print('Lecturas con variacion de 30 minutos:', int(work['variacion_temperatura_30min'].notna().sum()))
print('Lecturas sin variacion de 30 minutos:', int(work['variacion_temperatura_30min'].isna().sum()))
print('Motivo de la ausencia: no existe lectura previa dentro de [t-45, t-30] en el mismo viaje.')
print('Alguna base de variacion posterior a t:',
      int((work['ts_base_variacion'].notna() &
           (work['ts_base_variacion'] > work['ts_utc'])).sum()))
desfase = (work['ts_utc'] - work['ts_base_variacion']).dt.total_seconds().div(60)
print('Desfase de la base en minutos: min', float(desfase.min()), 'max', float(desfase.max()))
print('Lecturas fuera de la tolerancia [30-15, 30+15]:',
      int(((desfase < CONFIG['ventana_historica_min'] - CONFIG['tolerancia_busqueda_min']) |
           (desfase > CONFIG['ventana_historica_min'] + CONFIG['tolerancia_busqueda_min'])).sum()))

Regla:

 Busqueda temporal dentro del mismo viaje de la lectura mas reciente con timestamp en [t-45, t-30] minutos; variacion = temperatura actual menos temperatura de esa lectura. Sin lectura en ese rango la variacion queda nula.
Lecturas con variacion de 30 minutos: 26933
Lecturas sin variacion de 30 minutos: 1515
Motivo de la ausencia: no existe lectura previa dentro de [t-45, t-30] en el mismo viaje.
Alguna base de variacion posterior a t: 0
Desfase de la base en minutos: min 30.0 max 30.0
Lecturas fuera de la tolerancia [30-15, 30+15]: 0


In [7]:
def etiquetar_motivos_antecedente(df):
    out = df.copy()
    motivos = CONFIG['motivos']
    sin_anterior = out['ts_temperatura_anterior'].isna()
    out['motivo_temperatura_anterior'] = np.where(
        sin_anterior, motivos['primera_lectura'], motivos['con_antecedente'])
    sin_base = out['ts_base_variacion'].isna()
    out['motivo_variacion_30min'] = np.where(
        sin_base,
        np.where(sin_anterior, motivos['primera_lectura'], motivos['sin_lectura_30min']),
        motivos['con_antecedente'])
    return out


work = work.pipe(etiquetar_motivos_antecedente)

print('Motivos declarados en config:', CONFIG['motivos'])
print('Lecturas con temperatura anterior nula por motivo:',
      work.loc[work['motivo_temperatura_anterior'] != '', 'motivo_temperatura_anterior']
      .value_counts().to_dict())
print('Lecturas con variacion de 30 minutos nula por motivo:',
      work.loc[work['motivo_variacion_30min'] != '', 'motivo_variacion_30min']
      .value_counts().to_dict())
print('Nulos de temperatura_anterior sin motivo:',
      int((work['temperatura_anterior'].isna() & work['motivo_temperatura_anterior'].eq('')).sum()))
print('Nulos de variacion_temperatura_30min sin motivo:',
      int((work['variacion_temperatura_30min'].isna() & work['motivo_variacion_30min'].eq('')).sum()))
print('Valores de motivo en lecturas con derivado disponible (deben estar vacios):',
      int((work['temperatura_anterior'].notna() & work['motivo_temperatura_anterior'].ne('')).sum()),
      int((work['variacion_temperatura_30min'].notna() & work['motivo_variacion_30min'].ne('')).sum()))

Motivos declarados en config: {'con_antecedente': '', 'primera_lectura': 'primera_lectura_del_viaje', 'sin_lectura_30min': 'sin_lectura_en_[t-45,t-30]_minutos'}
Lecturas con temperatura anterior nula por motivo: {'primera_lectura_del_viaje': 1200}
Lecturas con variacion de 30 minutos nula por motivo: {'primera_lectura_del_viaje': 1200, 'sin_lectura_en_[t-45,t-30]_minutos': 315}
Nulos de temperatura_anterior sin motivo: 0
Nulos de variacion_temperatura_30min sin motivo: 0
Valores de motivo en lecturas con derivado disponible (deben estar vacios): 0 0


In [8]:
def construir_objetivo(df):
    out = df.copy()
    entidad = CONFIG['columnas']['entidad']
    horizonte = np.timedelta64(CONFIG['horizonte_objetivo_min'], 'm')
    minimo = np.timedelta64(
        CONFIG['horizonte_objetivo_min'] - CONFIG['tolerancia_cobertura_min'], 'm')
    objetivo = np.full(len(out), np.nan)
    evaluable = np.zeros(len(out), dtype=bool)
    for _, indices in out.groupby(entidad, sort=False).groups.items():
        indices = np.asarray(indices)
        ts_entidad = out['ts_utc'].dt.tz_convert('UTC').dt.tz_localize(None).to_numpy('datetime64[ns]')[indices]
        desviacion = out['desviacion_actual'].to_numpy()[indices]
        orden = np.argsort(ts_entidad, kind='stable')
        indices = indices[orden]
        ts_entidad = ts_entidad[orden]
        desviacion = desviacion[orden]
        acumulado = np.concatenate([[0.0], np.cumsum(np.nan_to_num(desviacion, nan=0.0))])
        siguientes = np.arange(len(ts_entidad)) + 1
        limite = np.searchsorted(ts_entidad, ts_entidad + horizonte, side='right')
        con_desviacion = (acumulado[limite] - acumulado[siguientes]) > 0
        con_cobertura = (limite > siguientes) & (ts_entidad[limite - 1] >= ts_entidad + minimo)
        objetivo[indices[con_cobertura]] = np.where(con_desviacion[con_cobertura], 1.0, 0.0)
        evaluable[indices] = con_cobertura
    out[CONFIG['columnas']['flag_silver']] = objetivo
    out['ventana_objetivo_evaluable'] = evaluable
    return out


work = work.pipe(construir_objetivo)
work['objetivo_coincide_con_silver'] = work[CONFIG['columnas']['flag_silver']].eq(work['flag_silver']) | (
    work[CONFIG['columnas']['flag_silver']].isna() & work['flag_silver'].isna())
work['calidad_estado'] = np.where(
    work['ventana_objetivo_evaluable'], 'objetivo_evaluable', 'objetivo_no_evaluable')
work['calidad_motivo'] = np.where(
    work['ventana_objetivo_evaluable'],
    'ventana_60min_observada_dentro_de_la_entidad',
    'sin_cobertura_futura_hasta_60min_en_la_entidad')

print('Regla del objetivo:', CONFIG['definicion_objetivo'])
print('Criterio de cobertura:', CONFIG['criterio_cobertura'])
print('Objetivo con ventana evaluable:')
print(work.loc[work['ventana_objetivo_evaluable'], CONFIG['columnas']['flag_silver']]
      .value_counts(dropna=False).to_string())
print('Objetivo nulo:', int(work[CONFIG['columnas']['flag_silver']].isna().sum()))
print('Objetivo nulo solo cuando la ventana no es evaluable:',
      bool(work[CONFIG['columnas']['flag_silver']].isna().equals(~work['ventana_objetivo_evaluable'])))
print('Etiquetas que coinciden con la columna que ya traia Silver:',
      int(work['objetivo_coincide_con_silver'].sum()),
      '| difieren:', int((~work['objetivo_coincide_con_silver']).sum()))

Regla del objetivo: 1 si existe al menos una desviacion_termica_flag igual a 1 en (t, t+60] dentro del mismo viaje; 0 solo con cobertura suficiente y sin desviacion; nulo sin cobertura suficiente.
Criterio de cobertura: La ventana objetivo es evaluable si la ultima lectura futura dentro de (t, t+60] alcanza al menos t+45 minutos. Sin esa cobertura el objetivo queda nulo y la bandera en False.
Objetivo con ventana evaluable:
desviacion_proximos_60min_flag
0.0    24477
1.0     1300
Objetivo nulo: 2671
Objetivo nulo solo cuando la ventana no es evaluable: True
Etiquetas que coinciden con la columna que ya traia Silver: 25774 | difieren: 2674


In [9]:
gold = (
    work[CONFIG['columnas_salida']]
    .sort_values([CONFIG['columnas']['entidad'], CONFIG['columnas']['timestamp']], kind='stable')
    .reset_index(drop=True)
)

print('Columnas de Gold:', len(gold.columns))
print(list(gold.columns))
print()
print(gold.head(8).to_string(index=False))

Columnas de Gold: 20
['_fila_bronze', 'timestamp', 'viaje_id', 'order_id', 'camion_id', 'producto_id', 'temperatura_cabina_c', 'humedad_cabina_pct', 'desviacion_termica_flag', 'temperatura_anterior', 'ts_temperatura_anterior', 'variacion_temperatura_30min', 'ts_base_variacion', 'motivo_temperatura_anterior', 'motivo_variacion_30min', 'desviacion_proximos_60min_flag', 'ventana_objetivo_evaluable', 'objetivo_coincide_con_silver', 'calidad_estado', 'calidad_motivo']

_fila_bronze                 timestamp  viaje_id       order_id camion_id producto_id temperatura_cabina_c humedad_cabina_pct desviacion_termica_flag  temperatura_anterior   ts_temperatura_anterior  variacion_temperatura_30min         ts_base_variacion motivo_temperatura_anterior    motivo_variacion_30min  desviacion_proximos_60min_flag  ventana_objetivo_evaluable  objetivo_coincide_con_silver     calidad_estado                               calidad_motivo
           2 2026-08-11 18:11:00+00:00 VIA-00001 ORD-2026-05710    CAM

In [10]:
ROL_PREDICTOR = {
    CONFIG['columnas']['temperatura']: 'predictor',
    CONFIG['columnas']['humedad']: 'predictor',
    CONFIG['columnas']['flag_actual']: 'predictor',
    'temperatura_anterior': 'predictor',
    'variacion_temperatura_30min': 'predictor',
}
ROL_IDENTIFICADOR = {
    CONFIG['columnas']['id_lectura']: 'identificador_lectura',
    CONFIG['columnas']['entidad']: 'identificador_entidad',
    CONFIG['columnas']['orden']: 'identificador_contexto',
    CONFIG['columnas']['camion']: 'identificador_contexto',
    CONFIG['columnas']['producto']: 'identificador_contexto',
    CONFIG['columnas']['timestamp']: 'marca_tiempo_prediccion',
}
ROL_OBJETIVO = {
    CONFIG['columnas']['flag_silver']: 'objetivo',
}
ROL_CONTROL = {
    'ts_temperatura_anterior': 'control_ventana',
    'ts_base_variacion': 'control_ventana',
    'ventana_objetivo_evaluable': 'control_cobertura',
}
ROL_AUDITORIA = {
    'motivo_temperatura_anterior': 'auditoria',
    'motivo_variacion_30min': 'auditoria',
    'objetivo_coincide_con_silver': 'auditoria',
    'calidad_estado': 'auditoria',
    'calidad_motivo': 'auditoria',
}
ROLES = {**ROL_IDENTIFICADOR, **ROL_PREDICTOR, **ROL_OBJETIVO, **ROL_CONTROL, **ROL_AUDITORIA}

faltantes_rol = [columna for columna in gold.columns if columna not in ROLES]
if faltantes_rol:
    raise ValueError(f'Columnas de Gold sin rol declarado: {faltantes_rol}')

clasificacion = pd.DataFrame({
    'columna': list(gold.columns),
    'rol': [ROLES[columna] for columna in gold.columns],
    'nulos': [int(gold[columna].isna().sum()) for columna in gold.columns],
    'ejemplo': [str(gold[columna].dropna().iloc[0]) if gold[columna].notna().any() else '' for columna in gold.columns],
})
print('Clasificacion de columnas de Gold')
print(clasificacion.to_string(index=False))
print()
print('Conteos por rol:', clasificacion['rol'].value_counts().to_dict())
PREDICTORES = [columna for columna, rol in ROLES.items() if rol == 'predictor']
OBJETIVOS = [columna for columna, rol in ROLES.items() if rol == 'objetivo']
NO_PREDICTABLES = [columna for columna, rol in ROLES.items()
                   if rol in ('objetivo', 'control_cobertura', 'control_ventana', 'auditoria')]
print('Predictores:', PREDICTORES)
print('Objetivo:', OBJETIVOS)
print('Columnas que el modelo NO debe usar como variable de entrada:', NO_PREDICTABLES)
print('Interseccion predictores con objetivo:', sorted(set(PREDICTORES) & set(OBJETIVOS)))
assert not set(PREDICTORES) & set(NO_PREDICTABLES), 'Fuga temporal: un predictor coincide con una columna de objetivo o auditoria'
print('Control de fuga por roles: OK')

Clasificacion de columnas de Gold
                       columna                     rol  nulos                                      ejemplo
                  _fila_bronze   identificador_lectura      0                                            2
                     timestamp marca_tiempo_prediccion      0                    2026-08-11 18:11:00+00:00
                      viaje_id   identificador_entidad      0                                    VIA-00001
                      order_id  identificador_contexto      0                               ORD-2026-05710
                     camion_id  identificador_contexto      0                                       CAM-20
                   producto_id  identificador_contexto      0                                     PROD-045
          temperatura_cabina_c               predictor      0                                         2.19
            humedad_cabina_pct               predictor      0                                         72.6
   

In [11]:
controles = []


def registrar(nombre, valor, detalle=''):
    controles.append({'control': nombre, 'resultado': valor, 'detalle': detalle})


ts_gold = pd.to_datetime(gold[CONFIG['columnas']['timestamp']], format='ISO8601', utc=True)
anterior_ts = pd.to_datetime(gold['ts_temperatura_anterior'], format='ISO8601', utc=True, errors='coerce')
base_ts = pd.to_datetime(gold['ts_base_variacion'], format='ISO8601', utc=True, errors='coerce')
objetivo = pd.to_numeric(gold[CONFIG['columnas']['flag_silver']], errors='coerce')
evaluable = gold['ventana_objetivo_evaluable'].astype(bool)

registrar('filas_silver', len(silver))
registrar('filas_gold', len(gold))
registrar('diferencia_filas', len(gold) - len(silver))
registrar('unicidad_clave_lectura', bool(not gold.duplicated(CONFIG['clave_lectura']).any()))
registrar('duplicados_clave_lectura', int(gold.duplicated(CONFIG['clave_lectura']).sum()))
registrar('id_lectura_unico', bool(gold[CONFIG['columnas']['id_lectura']].is_unique))
registrar('grupos_procesados', int(gold[CONFIG['columnas']['entidad']].nunique()))
registrar('grupos_entidad_silver', int(silver[CONFIG['columnas']['entidad']].nunique()))
registrar('timestamp_minimo', str(ts_gold.min()))
registrar('timestamp_maximo', str(ts_gold.max()))
registrar('grupos_con_orden_temporal_correcto',
          int(ts_gold.groupby(gold[CONFIG['columnas']['entidad']]).apply(
              lambda serie: bool(serie.is_monotonic_increasing)).sum()))
for columna in ['temperatura_anterior', 'ts_temperatura_anterior',
                'variacion_temperatura_30min', 'ts_base_variacion',
                CONFIG['columnas']['humedad']]:
    vacias = gold[columna].isna() | (gold[columna].astype(str) == '')
    registrar(f'nulos_{columna}', int(vacias.sum()))
registrar('objetivo_0', int((objetivo == 0).sum()))
registrar('objetivo_1', int((objetivo == 1).sum()))
registrar('objetivo_nulo', int(objetivo.isna().sum()))
registrar('ventanas_no_evaluables', int((~evaluable).sum()))
registrar('prevalencia_entre_evaluables', round(float((objetivo[evaluable] == 1).mean()), 6))
registrar('objetivo_cero_solo_con_cobertura',
          bool(objetivo[~evaluable].isna().all() and objetivo[evaluable].notna().all()))

referencia = pd.DataFrame({
    CONFIG['columnas']['entidad']: gold[CONFIG['columnas']['entidad']].to_numpy(),
    'ts': ts_gold.to_numpy(),
    'temp': pd.to_numeric(gold[CONFIG['columnas']['temperatura']], errors='coerce').to_numpy(),
})
temp_actual = pd.to_numeric(gold[CONFIG['columnas']['temperatura']], errors='coerce')
temp_ant_num = pd.to_numeric(gold['temperatura_anterior'], errors='coerce')
variacion_num = pd.to_numeric(gold['variacion_temperatura_30min'], errors='coerce')

cruce_anterior = gold.loc[anterior_ts.notna(), [CONFIG['columnas']['entidad'], 'ts_temperatura_anterior',
                                               'temperatura_anterior']].copy()
cruce_anterior = cruce_anterior.merge(
    referencia.rename(columns={'ts': 'ts_temperatura_anterior', 'temp': 'temp_esperada'}),
    on=[CONFIG['columnas']['entidad'], 'ts_temperatura_anterior'], how='left')
registrar('ventanas_no_mezclan_viajes_anterior', bool(cruce_anterior['temp_esperada'].notna().all()))
registrar('temperatura_anterior_coincide',
          bool(np.allclose(temp_ant_num[anterior_ts.notna()],
                           cruce_anterior['temp_esperada'].to_numpy())))

cruce_base = gold.loc[base_ts.notna(), [CONFIG['columnas']['entidad'], 'ts_base_variacion',
                                        CONFIG['columnas']['temperatura'],
                                        'variacion_temperatura_30min']].copy()
cruce_base = cruce_base.merge(
    referencia.rename(columns={'ts': 'ts_base_variacion', 'temp': 'temp_base'}),
    on=[CONFIG['columnas']['entidad'], 'ts_base_variacion'], how='left')
registrar('ventanas_no_mezclan_viajes_base', bool(cruce_base['temp_base'].notna().all()))
registrar('variacion_30min_coincide',
          bool(np.allclose(variacion_num[base_ts.notna()],
                           (temp_actual[base_ts.notna()] - cruce_base['temp_base'].to_numpy()))))

desfase = (ts_gold - base_ts).dt.total_seconds().div(60).dropna()
registrar('desfase_base_min_min', float(desfase.min()))
registrar('desfase_base_min_max', float(desfase.max()))
registrar('tolerancia_respetada',
          bool(desfase.between(CONFIG['ventana_historica_min'] - CONFIG['tolerancia_busqueda_min'],
                               CONFIG['ventana_historica_min'] + CONFIG['tolerancia_busqueda_min']).all()))

registrar('sin_predictores_posteriores_a_t',
          bool(anterior_ts.dropna().le(ts_gold[anterior_ts.notna()].to_numpy()).all()
               and base_ts.dropna().le(ts_gold[base_ts.notna()].to_numpy()).all()))
registrar('nulos_temperatura_anterior_sin_motivo',
          int((gold['temperatura_anterior'].isna() & gold['motivo_temperatura_anterior'].eq('')).sum()))
registrar('nulos_variacion_30min_sin_motivo',
          int((gold['variacion_temperatura_30min'].isna() & gold['motivo_variacion_30min'].eq('')).sum()))
registrar('motivo_temperatura_anterior_distribucion',
          gold.loc[gold['motivo_temperatura_anterior'] != '', 'motivo_temperatura_anterior']
          .value_counts().to_dict())
registrar('motivo_variacion_30min_distribucion',
          gold.loc[gold['motivo_variacion_30min'] != '', 'motivo_variacion_30min']
          .value_counts().to_dict())
registrar('predictores_sin_columnas_del_objetivo',
          not (set(PREDICTORES) & (set(OBJETIVOS) | {'ventana_objetivo_evaluable',
                                                    'objetivo_coincide_con_silver',
                                                    'calidad_estado', 'calidad_motivo'})))

verificacion = gold[[CONFIG['columnas']['entidad'], CONFIG['columnas']['timestamp'],
                     CONFIG['columnas']['flag_actual'], CONFIG['columnas']['flag_silver'],
                     'ventana_objetivo_evaluable']].copy()
verificacion['ts'] = ts_gold.to_numpy()
verificacion['flag_actual_num'] = pd.to_numeric(
    verificacion[CONFIG['columnas']['flag_actual']], errors='coerce')
grupo = verificacion.groupby(CONFIG['columnas']['entidad'], sort=False)
sig_1 = grupo['ts'].shift(-1)
sig_2_ts = grupo['ts'].shift(-2)
desv_1 = grupo['flag_actual_num'].shift(-1)
desv_2_flag = grupo['flag_actual_num'].shift(-2)
delta_1 = (sig_1 - verificacion['ts']).dt.total_seconds().div(60)
delta_2 = (sig_2_ts - verificacion['ts']).dt.total_seconds().div(60)
con_ventana = delta_1.notna() & (delta_1 > 0) & (delta_1 <= CONFIG['horizonte_objetivo_min'])
con_dos = con_ventana & delta_2.notna() & (delta_2 > 0) & (delta_2 <= CONFIG['horizonte_objetivo_min'])
esperado_flag = pd.Series(
    np.where(con_dos, np.maximum(desv_1, desv_2_flag), np.where(con_ventana, desv_1, np.nan)),
    index=verificacion.index, dtype='float64')
esperado_evaluable = con_ventana & (
    pd.Series(np.where(con_dos, delta_2, delta_1), index=verificacion.index, dtype='float64')
    >= CONFIG['horizonte_objetivo_min'] - CONFIG['tolerancia_cobertura_min'])
comparables = con_ventana & esperado_evaluable
ok_flag = bool(esperado_flag[comparables].eq(
    verificacion.loc[comparables, CONFIG['columnas']['flag_silver']]).all())
ok_evaluable = bool(esperado_evaluable[con_ventana].eq(
    verificacion.loc[con_ventana, 'ventana_objetivo_evaluable']).all())
ok_sin_cobertura_nula = bool(verificacion.loc[~esperado_evaluable,
                                              CONFIG['columnas']['flag_silver']].isna().all())
coincide = ok_flag and ok_evaluable and ok_sin_cobertura_nula
registrar('objetivo_recalculado_por_forma_independiente', coincide)
registrar('lecturas_con_ventana_comparable', int(comparables.sum()))

tabla_controles = pd.DataFrame(controles)
print('Controles finales')
print(tabla_controles.to_string(index=False))

Controles finales
                                     control                                                                      resultado detalle
                                filas_silver                                                                          28448        
                                  filas_gold                                                                          28448        
                            diferencia_filas                                                                              0        
                      unicidad_clave_lectura                                                                           True        
                    duplicados_clave_lectura                                                                              0        
                            id_lectura_unico                                                                           True        
                           grupos_procesados              

In [12]:
afirmaciones = {
    'filas silver igual a gold': len(silver) == len(gold),
    'sin diferencia de filas': len(gold) - len(silver) == 0,
    'clave de lectura unica': not gold.duplicated(CONFIG['clave_lectura']).any(),
    'identificador de lectura unico': gold[CONFIG['columnas']['id_lectura']].is_unique,
    'grupos completos': gold[CONFIG['columnas']['entidad']].nunique() == silver[CONFIG['columnas']['entidad']].nunique(),
    'objetivo con bandera coherente': bool(objetivo[evaluable].notna().all()
                                            and objetivo[~evaluable].isna().all()),
    'objetivo nunca cero sin cobertura': bool(objetivo[~evaluable].isna().all()),
    'temperatura anterior solo del pasado': bool(
        anterior_ts.dropna().le(ts_gold[anterior_ts.notna()].to_numpy()).all()),
    'base de variacion solo del pasado': bool(
        base_ts.dropna().le(ts_gold[base_ts.notna()].to_numpy()).all()),
    'sin mezcla de viajes en ventanas': bool(cruce_anterior['temp_esperada'].notna().all()
                                             and cruce_base['temp_base'].notna().all()),
    'objetivo coherente con calculo independiente': bool(coincide),
    'todo nulo de temperatura_anterior tiene motivo': bool(
        not (gold['temperatura_anterior'].isna() & gold['motivo_temperatura_anterior'].eq('')).any()),
    'todo nulo de variacion_30min tiene motivo': bool(
        not (gold['variacion_temperatura_30min'].isna() & gold['motivo_variacion_30min'].eq('')).any()),
    'motivo solo presente cuando el derivado falta': bool(
        not (gold['temperatura_anterior'].notna() & gold['motivo_temperatura_anterior'].ne('')).any()
        and not (gold['variacion_temperatura_30min'].notna() & gold['motivo_variacion_30min'].ne('')).any()),
    'motivos declarados en config': bool(
        set(gold.loc[gold['motivo_temperatura_anterior'] != '', 'motivo_temperatura_anterior'])
        <= {CONFIG['motivos']['primera_lectura']}
        and set(gold.loc[gold['motivo_variacion_30min'] != '', 'motivo_variacion_30min'])
        <= {CONFIG['motivos']['primera_lectura'], CONFIG['motivos']['sin_lectura_30min']}),
    'compuerta previa cumplida': errores_bloqueantes == 0,
    'sin predictores ni columnas de auditoria en el modelo': not (set(PREDICTORES) & set(NO_PREDICTABLES)),
}
fallidas = [nombre for nombre, valor in afirmaciones.items() if not valor]
print('Verificacion final:', 'CONTROLES_OK' if not fallidas else 'FALLOS: ' + ', '.join(fallidas))
if fallidas:
    raise AssertionError(f'Controles fallidos: {fallidas}')

Verificacion final: CONTROLES_OK


In [13]:
gold[CONFIG['columnas_salida']].to_csv(PATHS['gold'], index=False, encoding='utf-8')
print('Gold exportado en:', PATHS['gold'])
print('Archivo existe:', PATHS['gold'].exists())
print('Tamano en bytes:', PATHS['gold'].stat().st_size)

releido = pd.read_csv(PATHS['gold'], dtype=str, keep_default_na=False)
columnas_releidas = list(releido.columns)
print()
print('Filas leidas:', len(releido), '| filas escritas:', len(gold))
print('Columnas leidas:', len(releido.columns), '| columnas escritas:', len(gold.columns))
print('Columnas iguales:', list(releido.columns) == list(gold.columns))
print('Filas iguales:', len(releido) == len(gold))
re_ts = pd.to_datetime(releido[CONFIG['columnas']['timestamp']], format='ISO8601', utc=True)
re_objetivo = pd.to_numeric(releido[CONFIG['columnas']['flag_silver']], errors='coerce')
re_evaluable = releido['ventana_objetivo_evaluable'].str.lower().eq('true')
print('Tipos interpretables: timestamp', re_ts.dtype, '| objetivo float', re_objetivo.dtype)
print('Objetivo leido: 0 =', int((re_objetivo == 0).sum()),
      '| 1 =', int((re_objetivo == 1).sum()),
      '| nulo =', int(re_objetivo.isna().sum()))
print('Ventanas no evaluables leidas:', int((~re_evaluable).sum()))
print('Clave de lectura unica en el archivo leido:',
      not releido.duplicated(CONFIG['clave_lectura']).any())

Gold exportado en: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\datos\gold\andinalog_iot_modelado_gold.csv
Archivo existe: True
Tamano en bytes: 6868994

Filas leidas: 28448 | filas escritas: 28448
Columnas leidas: 20 | columnas escritas: 20
Columnas iguales: True
Filas iguales: True
Tipos interpretables: timestamp datetime64[us, UTC] | objetivo float float64
Objetivo leido: 0 = 24477 | 1 = 1300 | nulo = 2671
Ventanas no evaluables leidas: 2671
Clave de lectura unica en el archivo leido: True


In [14]:
releido['objetivo_num'] = pd.to_numeric(
    releido[CONFIG['columnas']['flag_silver']].replace('', np.nan), errors='coerce')
releido['evaluable_bool'] = releido['ventana_objetivo_evaluable'].str.lower().eq('true')
evaluable_re = releido['evaluable_bool']
objetivo_re = releido['objetivo_num']
temp_re = pd.to_numeric(releido[CONFIG['columnas']['temperatura']], errors='coerce')
humedad_re = pd.to_numeric(releido[CONFIG['columnas']['humedad']].replace('', np.nan), errors='coerce')
flag_actual_re = pd.to_numeric(releido[CONFIG['columnas']['flag_actual']], errors='coerce')
temp_anterior_re = pd.to_numeric(
    releido['temperatura_anterior'].replace('', np.nan), errors='coerce')
variacion_re = pd.to_numeric(
    releido['variacion_temperatura_30min'].replace('', np.nan), errors='coerce')

metricas = {
    'filas_silver': len(silver),
    'filas_gold': len(gold),
    'columnas_silver': int(len(silver.columns)),
    'columnas_gold': int(len(gold.columns)),
    'viajes': int(gold[CONFIG['columnas']['entidad']].nunique()),
    'ventana_evaluable': int(evaluable_re.sum()),
    'ventana_no_evaluable': int((~evaluable_re).sum()),
    'objetivo_1': int((objetivo_re == 1).sum()),
    'objetivo_0': int((objetivo_re == 0).sum()),
    'objetivo_nulo': int(objetivo_re.isna().sum()),
    'prevalencia_entre_evaluables': round(float((objetivo_re[evaluable_re] == 1).mean()), 6),
    'temperatura_anterior_disponible': int(temp_anterior_re.notna().sum()),
    'variacion_30min_disponible': int(variacion_re.notna().sum()),
    'humedad_ausente': int(humedad_re.isna().sum()),
    'desviacion_actual_1': int((flag_actual_re == 1).sum()),
    'objetivo_coincide_con_silver': int(releido['objetivo_coincide_con_silver'].str.lower().eq('true').sum()),
    'objetivo_difiere_de_silver': int(releido['objetivo_coincide_con_silver'].str.lower().eq('false').sum()),
    'motivo_temperatura_anterior_primera': int(releido['motivo_temperatura_anterior']
                                              .eq(CONFIG['motivos']['primera_lectura']).sum()),
    'motivo_variacion_primera': int(releido['motivo_variacion_30min']
                                    .eq(CONFIG['motivos']['primera_lectura']).sum()),
    'motivo_variacion_sin_lectura': int(releido['motivo_variacion_30min']
                                        .eq(CONFIG['motivos']['sin_lectura_30min']).sum()),
}

config_esperada = CONFIG['frecuencia_esperada_min']
try:
    clasificacion_texto = clasificacion.to_markdown(index=False)
except Exception:
    clasificacion_texto = chr(10).join(['```', clasificacion.to_string(index=False), '```'])

reporte = f"""# Informe S2G 01 - AndinaLog IoT Gold

## Objetivo del producto Gold
Construir la tabla base del modelo predictivo de 03B: una fila por lectura de telemetria que
permita predecir si ocurrira una desviacion termica durante los proximos 60 minutos
(`desviacion_proximos_60min_flag`).

## Fuente utilizada
- Entrada unica: `{CONFIG['rutas']['silver']}`.
- No se realizo ningun join: IoT Silver contiene todas las variables minimas.
- Fuentes contextuales o excluidas: ninguna. El producto no necesita WMS Orders, flota, inventario,
  HR Drivers ni Bitacora.

## Compuerta previa y evidencia de la fuente
La skill de Gold exige como entrada minima IoT Silver con evidencia de conciliacion y de auditoria.
El notebook verifica esta compuerta con los archivos actuales, antes de construir Gold, y falla si
no se cumple.
- Bronze de origen: `{CONFIG['evidencia_entrada']['bronze']}` ({len(bronze)} filas leidas en esta ejecucion).
- Silver derivado: `{CONFIG['rutas']['silver']}` ({len(silver)} filas leidas en esta ejecucion).
- Cuarentena de la etapa Bronze-Silver: `{CONFIG['evidencia_entrada']['cuarentena']}` ({len(cuarentena)} filas leidas en esta ejecucion).
- Informe de la etapa Bronze-Silver: `{CONFIG['evidencia_entrada']['informe_silver']}`.
- Notebook de la etapa Bronze-Silver: `{CONFIG['evidencia_entrada']['notebook_silver']}`.
- Conciliacion verificada en esta ejecucion: Bronze {len(bronze)} = Silver {len(silver)} + cuarentena {len(cuarentena)}.
- Conciliacion declarada en el informe de la etapa Bronze-Silver: {conciliacion.group(0) if conciliacion else 'no encontrada'}.
- Silver sin errores bloqueantes: {errores_bloqueantes}. Silver sin lecturas imputadas: {int(silver['fue_imputada'].eq('True').sum()) if 'fue_imputada' in silver.columns else 0}.
- Clave de lectura de Silver unica: {not silver.duplicated(CONFIG['clave_lectura']).any()}.
- La etapa Bronze-Silver de esta fuente tiene su propia auditoria; la evidencia citada arriba es la
  que sustenta el uso de Silver como base de este producto, y no se modifica ninguna decision de esa etapa.

## Unidad de observacion
Una lectura de telemetria. Una fila Gold por cada fila Silver, sin agregar ni muestreo.

## Esquema real encontrado en IoT Silver
- Filas: {len(silver)}. Columnas: {len(silver.columns)}.
- Viajes distintos: {silver[CONFIG['columnas']['entidad']].nunique()}.
- Ordenes distintos: {silver[CONFIG['columnas']['orden']].nunique()}; camiones distintos: {silver[CONFIG['columnas']['camion']].nunique()}.
- Timestamp con sufijo `+00:00` en todas las filas, es decir UTC; rango de
  {pd.to_datetime(silver[CONFIG['columnas']['timestamp']], format='ISO8601', utc=True).min()} a
  {pd.to_datetime(silver[CONFIG['columnas']['timestamp']], format='ISO8601', utc=True).max()}.
- Identificador de lectura disponible: `{CONFIG['columnas']['id_lectura']}`, unico en las {len(silver)} filas.
- Frecuencia observada dentro de cada viaje: intervalo mediano
  {float(gaps.median()):.0f} minutos, con intervalos de 30, 60 y 90 minutos; ninguna lectura con
  intervalo no multiple de la cadencia. La cadencia no es completamente regular, por lo que las
  ventanas se resuelven con tiempo y no con desplazamiento fijo de filas.
- Vacios heredados: humedad ausente en {int(silver[CONFIG['columnas']['humedad']].eq('').sum())} lecturas;
  temperatura ausente en {int(pd.to_numeric(silver[CONFIG['columnas']['temperatura']].replace('', np.nan), errors='coerce').isna().sum())} lecturas.
- Silver no trae errores bloqueantes ni lecturas imputadas.

## Clave de agrupacion
`{CONFIG['columnas']['entidad']}` (`{CONFIG['columnas']['entidad']}` con patron `{CONFIG['patron_entidad']}`).
Clave de lectura: `{CONFIG['columnas']['entidad']}` mas `timestamp`. Ninguna ventana cruza viajes,
ordenes o camiones.

## Definicion de cada predictor
| Predictor | Definicion |
|---|---|
| `{CONFIG['columnas']['temperatura']}` | Temperatura de la cabina en la lectura actual, tal como entrega Silver. |
| `{CONFIG['columnas']['humedad']}` | Humedad relativa de la cabina en la lectura actual; nula si Silver la trae vacia, sin imputar. |
| `{CONFIG['columnas']['flag_actual']}` | Indicador de desviacion termica observada en el instante t, tal como entrega Silver. |
| `temperatura_anterior` | {CONFIG['metodo_temperatura_anterior']} |
| `variacion_temperatura_30min` | {CONFIG['metodo_variacion']} |

Columnas de control de ventana: `ts_temperatura_anterior` y `ts_base_variacion` guardan el instante
exacto de la lectura usada como antecedente, para demostrar que ningun predictor usa el futuro.

## Definicion exacta del objetivo
{CONFIG['definicion_objetivo']}

La desviacion del instante t no se incluye como evento futuro: el intervalo evaluado es
`(t, t+60]` de forma estricta.

## Metodo para la variacion de 30 minutos
{CONFIG['metodo_variacion']}

La tolerancia de {CONFIG['tolerancia_busqueda_min']} minutos es la mitad de la cadencia observada de
{config_esperada} minutos, de modo que la busqueda nunca alcanza una lectura mas antigua que t-30
y nunca puede tomar informacion posterior a t. Cuando el intervalo previo real es de 60 o 90
minutos, no existe lectura en el rango y la variacion queda nula, con el motivo registrado en
`motivo_variacion_30min`. No se usa `shift` fijo por filas ni `merge_asof` sin verificar la frecuencia.

## Criterio de cobertura futura
{CONFIG['criterio_cobertura']}

Es decir, la ventana es evaluable cuando existe una lectura con instante entre `t+45` y `t+60`
minutos dentro del mismo viaje. Con cadencia de 30 minutos esto equivale a exigir la lectura de
`t+60`. Las lecturas finales de cada viaje quedan con objetivo nulo y
`ventana_objetivo_evaluable=False`; nunca se rellenan con cero.

## Conteos de filas y grupos
- Filas Silver: {len(silver)}. Filas Gold: {len(gold)}. Diferencia: {len(gold) - len(silver)}.
- Viajes: {metricas['viajes']}, iguales a los viajes de Silver.
- Columnas Silver: {len(silver.columns)}. Columnas Gold: {len(gold.columns)}.

## Distribucion del objetivo
- Ventanas evaluables: {metricas['ventana_evaluable']}.
- Ventanas no evaluables: {metricas['ventana_no_evaluable']}.
- Objetivo 1: {metricas['objetivo_1']}. Objetivo 0: {metricas['objetivo_0']}. Objetivo nulo: {metricas['objetivo_nulo']}.
- Prevalencia entre ventanas evaluables: {metricas['prevalencia_entre_evaluables']}.
- Lecturas con desviacion actual: {metricas['desviacion_actual_1']} de {len(gold)}.

## Cobertura de predictores derivados
- `temperatura_anterior` disponible en {metricas['temperatura_anterior_disponible']} de {len(gold)} lecturas.
- `variacion_temperatura_30min` disponible en {metricas['variacion_30min_disponible']} de {len(gold)} lecturas.
- `humedad_cabina_pct` ausente en {metricas['humedad_ausente']} lecturas; se conserva nula, sin imputar.
- No se imputo ningun predictor ni identificador.

## Motivos de los nulos de predictores derivados
Todo nulo de un predictor derivado lleva su motivo en una columna propia; el motivo aparece solo
cuando el derivado falta, y el control final lo verifica.
| Columna de motivo | Motivo | Lecturas |
|---|---|---|
| `motivo_temperatura_anterior` | {CONFIG['motivos']['primera_lectura']} | {metricas['motivo_temperatura_anterior_primera']} |
| `motivo_variacion_30min` | {CONFIG['motivos']['primera_lectura']} | {metricas['motivo_variacion_primera']} |
| `motivo_variacion_30min` | {CONFIG['motivos']['sin_lectura_30min']} | {metricas['motivo_variacion_sin_lectura']} |
- `temperatura_anterior` nula en {len(gold) - metricas['temperatura_anterior_disponible']} lecturas, todas con motivo.
- `variacion_temperatura_30min` nula en {len(gold) - metricas['variacion_30min_disponible']} lecturas, todas con motivo.
- El motivo de cobertura del objetivo se conserva aparte en `calidad_estado` y `calidad_motivo`; no se
  mezcla con el motivo de los antecedentes.

## Uso posterior de la tabla: variables admitidas y prohibidas
- Predictores que el EDA y el modelo deben usar: {', '.join(PREDICTORES)}.
- Variable objetivo: {CONFIG['columnas']['flag_silver']}.
- Columnas que NO deben usarse como variable de entrada, por ser de objetivo, control o auditoria:
  {', '.join(NO_PREDICTABLES)}.
- No se debe usar "todas las columnas salvo el objetivo": las {len(NO_PREDICTABLES)} columnas de objetivo,
  control y auditoria incorporarian informacion derivada del objetivo y producirian fuga temporal; el
  notebook verifica que el conjunto de predictores y el conjunto de columnas no admisibles son disjuntos.

## Controles contra fuga temporal
- `ts_temperatura_anterior` y `ts_base_variacion` son menores o iguales que `timestamp` en todas las lecturas.
- Los predictores no incluyen `desviacion_proximos_60min_flag` ni ninguna columna de objetivo, control o
  auditoria; la lista completa de columnas no admisibles esta en la seccion de uso posterior.
- El objetivo se recalculo con una forma independiente basada en `shift` y se comparo fila a fila.
- Toda ventana se resolvio dentro de `{CONFIG['columnas']['entidad']}`: la temperatura anterior y la base de
  variacion se recuperaron uniendo por viaje y por instante, y todas las ventanas se resolvieron contra su propio viaje.
- La tabla de roles y la lista de columnas no admisibles como entrada estan en la seccion de clasificacion.

## Clasificacion de columnas de Gold
{clasificacion_texto}

## Resultados de conciliacion
- Filas Silver = Filas Gold: {len(silver)} = {len(gold)}; diferencia {len(gold) - len(silver)}.
- Sin eliminacion de filas: no se uso `dropna`; las lecturas sin antecedente o sin cobertura se conservan.
- Sin multiplicacion: una fila Gold por lectura Silver, clave de lectura unica.
- CSV exportado con codificacion UTF-8 y sin indice; relectura verificada con {len(releido)} filas y
  {len(columnas_releidas)} columnas, y tipos interpretables.
- Etiqueta de Gold identica a la que traia Silver en {metricas['objetivo_coincide_con_silver']} lecturas y
  distinta en {metricas['objetivo_difiere_de_silver']}. La diferencia proviene de que Silver no expone
  la falta de cobertura como objetivo nulo, mientras que Gold si lo hace.

## Limitaciones
- La cadencia observada es de 30 minutos, por lo que la ventana de 30 minutos se resuelve con una sola
  lectura base y no con un promedio; los intervalos reales de 60 y 90 minutos dejan variacion nula.
- Las ultimas lecturas de cada viaje no tienen ventana de 60 minutos observada y quedan con objetivo nulo.
- La etiqueta describe asociacion temporal con una desviacion posterior; no afirma causalidad ni
  anticipa reglas de negocio, y no se usa para sancionar conductores.
- La evaluacion de cobertura admite una ventana observada de al menos 45 de los 60 minutos; lecturas
  adicionales dentro de la ventana pueden faltar en viajes con huecos.
- La tabla no incluye variables de otras fuentes; el alcance minimo de 03B no las exige.

## Rutas de los entregables
- Notebook: `{CONFIG['rutas']['notebook']}`
- Gold: `{CONFIG['rutas']['gold']}`
- Informe: `{CONFIG['rutas']['informe']}`

## Reproducibilidad
- Ejecucion UTC: {EJECUTADO_UTC}.
- Python: {platform.python_version()}.
- pandas: {pd.__version__}.
- numpy: {np.__version__}.
- El pipeline es determinista y no usa aleatoriedad, por lo que no se declara semilla.
- Ejecutar las celdas en orden desde la raiz del proyecto; `detectar_raiz()` busca la raiz entre el
  directorio actual y sus ancestros, o la variable de entorno `ANDINALOG_ROOT`.
- Los controles finales se aplican sobre el CSV Gold persistido, no solo sobre el dataframe en memoria.
"""

PATHS['informe'].write_text(reporte, encoding='utf-8')
print('Informe generado en:', PATHS['informe'])
print('Archivo existe:', PATHS['informe'].exists())
print()
print(json.dumps(metricas, ensure_ascii=False, indent=2))

Informe generado en: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\informes\silver_gold\Informe_S2G_01_IoT_Gold.md
Archivo existe: True

{
  "filas_silver": 28448,
  "filas_gold": 28448,
  "columnas_silver": 78,
  "columnas_gold": 20,
  "viajes": 1200,
  "ventana_evaluable": 25777,
  "ventana_no_evaluable": 2671,
  "objetivo_1": 1300,
  "objetivo_0": 24477,
  "objetivo_nulo": 2671,
  "prevalencia_entre_evaluables": 0.050433,
  "temperatura_anterior_disponible": 27248,
  "variacion_30min_disponible": 26933,
  "humedad_ausente": 98,
  "desviacion_actual_1": 940,
  "objetivo_coincide_con_silver": 25774,
  "objetivo_difiere_de_silver": 2674,
  "motivo_temperatura_anterior_primera": 1200,
  "motivo_variacion_primera": 1200,
  "motivo_variacion_sin_lectura": 315
}
